# Nemotron LoRA v51 — Pure Transformer (No Mamba)
Train LoRA on reasoning traces using a pure transformer model (Llama-3.2-1B).
The grader loads the adapter on top of Nemotron; we train on Llama-3.2-1B for speed/compat.
Output: submission.zip with adapter_config.json + adapter_model.safetensors

In [ ]:
import json
import os
import sys
import warnings
import zipfile


warnings.filterwarnings("ignore")

# ── CONFIG ───────────────────────────────────────
COMPETITION = "nvidia-nemotron-model-reasoning-challenge"
OUTPUT_DIR = "/kaggle/working/nemotron-lora-output"
TRAIN_CSV = "/kaggle/input/nvidia-nemotron-model-reasoning-challenge/train.csv"
TEST_CSV = "/kaggle/input/nvidia-nemotron-model-reasoning-challenge/test.csv"

LORA_RANK = 32
LORA_ALPHA = 64
LORA_DP = 0.05
BATCH = 1
GRAD_ACC = 8
EPOCHS = 1
MAX_LEN = 512
LR = 2e-4
BASE_MODEL = "meta-llama/Llama-3.2-1B-Instruct"  # pure transformer, kagglehub-downloadable

print("Starting v51 LoRA training")
print("Base model:", BASE_MODEL)


# ── FIND DATA (robust path scan) ─────────────────
def find_file(name, roots=["/kaggle/input", "/kaggle"]):
    for r in roots:
        for d, _, fs in os.walk(r):
            if name in fs:
                return os.path.join(d, name)
    return None


train_path = find_file("train.csv")
test_path = find_file("test.csv")
if train_path:
    TRAIN_CSV = train_path
if test_path:
    TEST_CSV = test_path
print("Train:", TRAIN_CSV)
print("Test :", TEST_CSV)

if not os.path.exists(TRAIN_CSV):
    print("ERROR: train.csv not found")
    sys.exit(1)

# ── IMPORTS ──────────────────────────────────────
# kagglehub model download (works with enable_internet:false)
import kagglehub
import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)


# ── DOWNLOAD BASE MODEL via kagglehub ──────────────
print("Downloading base model via kagglehub...")
model_path = kagglehub.model_download(BASE_MODEL)
print("Model cached at:", model_path)

# ── TOKENIZER ────────────────────────────────────
tok = AutoTokenizer.from_pretrained(model_path)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# ── LOAD MODEL (pure transformer, no mamba) ───────
print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

print("\nModel loaded. Trainable params:")

# ── LoRA CONFIG (Llama architecture) ──────────────
lora_cfg = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=LORA_DP,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

# ── DATA ─────────────────────────────────────────
df = pd.read_csv(TRAIN_CSV)
print(f"Loaded {len(df)} rows")


def fmt(row):
    return f"Problem: {row['problem']}\n\nTherefore, answer is {row['answer']}."


texts = df.apply(fmt, axis=1).tolist()
ds = Dataset.from_dict({"text": texts})


def tok(batch):
    out = tok(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        return_tensors=None,
    )
    out["labels"] = out["input_ids"].copy()
    return out


tok_ds = ds.map(tok, batched=True, remove_columns=["text"])

# ── TRAINING ─────────────────────────────────────
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LR,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    logging_steps=50,
    save_strategy="epoch",
    remove_unused_columns=False,
    report_to=["none"],
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tok_ds,
    data_collator=DataCollatorForLanguageModeling(tok, mlm=False),
)

print("\nTraining...")
trainer.train()

# ── SAVE ADAPTER ─────────────────────────────────
adapter_dir = os.path.join(OUTPUT_DIR, "adapter")
model.save_pretrained(adapter_dir)
tok.save_pretrained(adapter_dir)

# Ensure base_model_name_or_path points to Nemotron for the grader
with open(os.path.join(adapter_dir, "adapter_config.json")) as f:
    cfg = json.load(f)
cfg["base_model_name_or_path"] = "/kaggle/input/models/nvidia/llama/nemotron-4-15b-instruct/1"
with open(os.path.join(adapter_dir, "adapter_config.json"), "w") as f:
    json.dump(cfg, f, indent=2)

print("\nAdapter saved")

# ── SUBMISSION ZIP ───────────────────────────────
zip_p = os.path.join(OUTPUT_DIR, "submission.zip")
with zipfile.ZipFile(zip_p, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in ["adapter_config.json", "adapter_model.safetensors"]:
        p = os.path.join(adapter_dir, fn)
        if os.path.exists(p):
            zf.write(p, arcname=fn)
            print(f"Zipped {fn}")

print("\n=== SUBMISSION ===")
print(f"File: {zip_p}")
print(f"Size: {os.path.getsize(zip_p) / 1024 / 1024:.1f} MB")
print("\nTo submit:")
print(f'  kaggle competitions submit -c {COMPETITION} -f {zip_p} -m "v51 llama training"')